In [ ]:
from os import getenv

import pandas as pd
import requests
import country_converter as coco
# import folium
from bs4 import BeautifulSoup
from sqlalchemy import create_engine

In [2]:
engine = create_engine(getenv("SQLALCHEMY_RELOHELPER_URL"))

## Этап 2 - работа с Numbeo
Подготовка большого df, содержащего всю информацию по городам. (сохранение df `.pkl`)

Следующий этап - разделение df на таблицы под БД и загрузка в БД.

In [3]:
url = "https://www.numbeo.com/cost-of-living/rankings_current.jsp"
df = pd.read_html(url)[1]
df.head(2)

,Rank,City,Cost of Living Index,Rent Index,Cost of Living Plus Rent Index,Groceries Index,Restaurant Price Index,Local Purchasing Power Index
0,NaN,"Zurich, Switzerland",123.9,71.3,99.6,125.6,128.2,176.4
1,NaN,"Geneva, Switzerland",118.4,64.3,93.4,116.0,126.0,161.3


In [4]:
indexes_dict = {
    "https://www.numbeo.com/quality-of-life/rankings_current.jsp": [
        "City",
        "Quality of Life Index",
        "Property Price to Income Ratio",
        "Traffic Commute Time Index",
        "Climate Index",
    ],
    "https://www.numbeo.com/crime/rankings_current.jsp": ["City", "Safety Index"],
    "https://www.numbeo.com/health-care/rankings_current.jsp": [
        "City",
        "Health Care Index",
    ],
    "https://www.numbeo.com/pollution/rankings_current.jsp": [
        "City",
        "Pollution Index",
    ],
}


In [5]:
for key, value in indexes_dict.items():
    df_temp = pd.read_html(key)[1][value]
    df = df.merge(df_temp, how="left", on="City")

df.drop(["Rank", "Restaurant Price Index"], axis=1, inplace=True)
df.columns = list(
    map(lambda x: x.removesuffix(" Index").lower().replace(" ", "_"), df.columns)
)
df.rename(columns={"city": "city_country"}, inplace=True)
# df.head(3)


In [6]:
df.head(5)

,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution
0,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3
1,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2
2,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4
3,"Lugano, Switzerland",113.8,44.2,81.6,109.9,160.3,NaN,NaN,NaN,NaN,78.4,NaN,NaN
4,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,181.2,206.4,11.0,33.3,73.3,71.6,70.3,NaN


In [7]:
response = requests.get(url).text
soup = BeautifulSoup(response, "lxml")
links = soup.find_all("td", class_="cityOrCountryInIndicesTable")

href_list = []
for i in links:
    href_list.append(
        [i.find("a").text, i.find("a").get("href") + "?displayCurrency=USD"]
    )

city_link = pd.DataFrame(href_list, columns=["city_country", "link"])

df = df.merge(city_link, how="left", on="city_country")


In [8]:
split_table1 = df["city_country"].str.rsplit(", ", n=1, expand=True)
split_table1.columns = ["city_state", "country"]

split_table2 = split_table1["city_state"].str.split(", ", n=1, expand=True)
split_table2.columns = ["city", "state_code"]

df = pd.concat([split_table2, split_table1["country"], df], axis=1)

df["city"] = df.apply(
    lambda row: row.city.split(" (")[0], axis=1
)  # removing duplicate cities in brackets

# df.head()


In [9]:
df = df.drop(df.loc[df["country"] == "Kosovo (Disputed Territory)"].index)


In [10]:
new_df = pd.DataFrame({"country": df["country"].unique()})
new_df["country_code"] = new_df.apply(
    lambda row: coco.convert(names=row.country, to="ISO3"), axis=1
)
df = df.merge(new_df, how="left", on="country")

# df['country_code'] = df.apply(lambda row: coco.convert(names=row.country, to='ISO3') , axis = 1) - can be on one line, but it takes 4 times as long
df.head(3)


,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code
0,Zurich,NaN,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...,CHE
1,Geneva,NaN,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...,CHE
2,Basel,NaN,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...,CHE


In [11]:
df[df["state_code"].notnull() & (df["country"] != "United States")]

,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code
177,Lethbridge,AB,Canada,"Lethbridge, AB, Canada",66.4,20.9,45.4,69.2,131.6,NaN,NaN,NaN,NaN,36.9,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Lethb...,CAN
198,St. John's,Newfoundland and Labrador,Canada,"St. John's, Newfoundland and Labrador, Canada",64.8,21.7,44.9,73.4,127.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.numbeo.com/cost-of-living/in/St-Jo...,CAN
222,Nanaimo,BC,Canada,"Nanaimo, BC, Canada",63.2,30.4,48.0,73.5,127.8,188.0,5.9,26.0,92.3,46.8,61.4,NaN,https://www.numbeo.com/cost-of-living/in/Nanai...,CAN
468,Batumi,Ajara,Georgia,"Batumi, Ajara, Georgia",30.5,10.7,21.4,30.1,52.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Batum...,GEO
474,Qingdao,Shandong,China,"Qingdao, Shandong, China",29.6,8.2,19.7,36.9,104.7,177.5,13.6,27.9,70.4,NaN,76.7,50.5,https://www.numbeo.com/cost-of-living/in/Qingd...,CHN


#### В df обнулить `state_code`, если `country_code` != 'USA'

In [12]:
df.loc[df["country_code"] != "USA", "state_code"] = None

Снова проверка

In [13]:
df[df["state_code"].notnull() & (df["country"] != "United States")]

,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code


In [14]:
df[df["country_code"] == "USA"].head()

,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code
7,Honolulu,HI,United States,"Honolulu, HI, United States",101.7,61.1,83.0,121.3,96.6,162.9,9.7,40.4,95.3,52.6,73.4,37.1,https://www.numbeo.com/cost-of-living/in/Honol...,USA
8,New York,NY,United States,"New York, NY, United States",100.0,100.0,100.0,100.0,100.0,133.6,14.3,43.5,79.7,49.1,62.8,58.1,https://www.numbeo.com/cost-of-living/in/New-Y...,USA
10,San Francisco,CA,United States,"San Francisco, CA, United States",96.7,75.8,87.0,104.2,160.5,171.5,7.0,49.0,97.3,39.4,64.9,49.0,https://www.numbeo.com/cost-of-living/in/San-F...,USA
15,Seattle,WA,United States,"Seattle, WA, United States",89.2,57.8,74.7,97.1,168.6,188.8,5.0,42.9,91.7,44.8,66.7,38.8,https://www.numbeo.com/cost-of-living/in/Seatt...,USA
17,San Jose,CA,United States,"San Jose, CA, United States",88.2,70.3,80.0,87.9,135.1,173.1,8.4,38.3,95.5,52.0,68.0,48.4,https://www.numbeo.com/cost-of-living/in/San-J...,USA


### Приведение кодов США к виду `US-NY`, вместо `NY`

In [15]:
# indexes = df[df["state_code"].notnull() & (df["country"] != "United States")][
#     "state_code"
# ].index
# df.loc[indexes, "state_code"] = None

# df["state_code"] = df.apply(
#     lambda row: f"US-{row['state_code']}" if row["state_code"] != None else None, axis=1
# )


1. Проверка: нет ли пустых state_code у США

In [16]:
df_us_missing_state = df[(df["country"] == "United States") & (df["state_code"].isna())]

print("US cities without state_code:", len(df_us_missing_state))

display(df_us_missing_state)

US cities without state_code: 0


,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code


2. Проверка: нет ли state_code у НЕ-США

In [17]:
df_non_us_with_state = df[
    (df["country"] != "United States") & (df["state_code"].notna())
]

print("Non-US cities with state_code:", len(df_non_us_with_state))

display(df_non_us_with_state)

Non-US cities with state_code: 0


,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code


3. После проверок делаем преобразование

In [18]:
mask = (df["country"] == "United States") & (df["state_code"].notna())

df.loc[mask, "state_code"] = "US-" + df.loc[mask, "state_code"]


4. Финальная проверка

In [19]:
df[(df["country"] == "United States") & (~df["state_code"].str.startswith("US-"))]


,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code


In [20]:
invalid_us_state = df[
    (df["country"] == "United States")
    & (~df["state_code"].fillna("").str.startswith("US-"))
]

print("Invalid US state_code rows:", len(invalid_us_state))
display(invalid_us_state)

Invalid US state_code rows: 0


,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code


## Map

In [21]:
# map_df = (
#     df.groupby("country_code")["city"]
#     .nunique()
#     .reset_index()
#     .sort_values("city", ascending=False)
# )

# m = folium.Map(location=[37.87820990704326, 6.555063556986549], zoom_start=1.5)
# folium.Choropleth(
#     geo_data="./data/world.geojson",
#     name="choropleth",
#     data=map_df,
#     columns=["country_code", "city"],
#     key_on="feature.properties.ISO_A3",
#     fill_color="YlGn",
#     nan_fill_color="pink",
#     fill_opacity=0.8,
#     bins=7,
#     reset=True,
#     highlight=True,
#     legend_name="Count of cities",
# ).add_to(m)

In [22]:
# m

### Пока решил не удалять Индию

In [23]:
# india = df.loc[
#     (df["country"] == "India") & (df["city"] != "Delhi") & (df["city"] != "Mumbai")
# ]

# df = df.drop(india.index)

In [24]:
df.head(3)

,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code
0,Zurich,NaN,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...,CHE
1,Geneva,NaN,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...,CHE
2,Basel,NaN,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...,CHE


#### Работа с индексами `city_id`

In [25]:
df.reset_index(drop=True, inplace=True)
df.insert(0, "city_id", df.index + 1)
df.set_index("city_id", inplace=True)
df.head(3)

,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code
city_id,,,,,,,,,,,,,,,,,,
1,Zurich,NaN,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...,CHE
2,Geneva,NaN,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...,CHE
3,Basel,NaN,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...,CHE


### data 2026 !

In [26]:
states_tbl = pd.read_sql("state", engine).loc[:, ["state_code", "state_name"]]
df = df.merge(states_tbl, on="state_code", how="left")

In [27]:
df.head(15)

,city,state_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link,country_code,state_name
0,Zurich,NaN,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...,CHE,NaN
1,Geneva,NaN,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...,CHE,NaN
2,Basel,NaN,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...,CHE,NaN
3,Lugano,NaN,Switzerland,"Lugano, Switzerland",113.8,44.2,81.6,109.9,160.3,NaN,NaN,NaN,NaN,78.4,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Lugan...,CHE,NaN
4,Lausanne,NaN,Switzerland,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,181.2,206.4,11.0,33.3,73.3,71.6,70.3,NaN,https://www.numbeo.com/cost-of-living/in/Lausa...,CHE,NaN
5,George Town,NaN,Cayman Islands,"George Town, Cayman Islands",110.5,74.6,93.9,117.0,159.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Georg...,CYM,NaN
6,Bern,NaN,Switzerland,"Bern, Switzerland",108.6,43.9,78.7,109.8,182.3,208.1,9.5,38.8,76.0,74.7,69.3,26.6,https://www.numbeo.com/cost-of-living/in/Bern?...,CHE,NaN
7,Honolulu,US-HI,United States,"Honolulu, HI, United States",101.7,61.1,83.0,121.3,96.6,162.9,9.7,40.4,95.3,52.6,73.4,37.1,https://www.numbeo.com/cost-of-living/in/Honol...,USA,Hawaii
8,New York,US-NY,United States,"New York, NY, United States",100.0,100.0,100.0,100.0,100.0,133.6,14.3,43.5,79.7,49.1,62.8,58.1,https://www.numbeo.com/cost-of-living/in/New-Y...,USA,New York
9,Reykjavik,NaN,Iceland,"Reykjavik, Iceland",98.7,47.2,74.9,106.9,112.7,196.6,8.0,20.5,68.8,75.5,69.6,15.6,https://www.numbeo.com/cost-of-living/in/Reykj...,ISL,NaN


In [29]:
new_order = [
    "city",
    "state_code",
    "state_name",
    "country_code",
    "country",
    "city_country",
    "cost_of_living",
    "rent",
    "cost_of_living_plus_rent",
    "groceries",
    "local_purchasing_power",
    "quality_of_life",
    "property_price_to_income_ratio",
    "traffic_commute_time",
    "climate",
    "safety",
    "health_care",
    "pollution",
    "link",
]

df = df[new_order].copy()

In [32]:
df

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link
0,Zurich,NaN,NaN,CHE,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...
1,Geneva,NaN,NaN,CHE,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...
2,Basel,NaN,NaN,CHE,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...
3,Lugano,NaN,NaN,CHE,Switzerland,"Lugano, Switzerland",113.8,44.2,81.6,109.9,160.3,NaN,NaN,NaN,NaN,78.4,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Lugan...
4,Lausanne,NaN,NaN,CHE,Switzerland,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,181.2,206.4,11.0,33.3,73.3,71.6,70.3,NaN,https://www.numbeo.com/cost-of-living/in/Lausa...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,Indore,NaN,NaN,IND,India,"Indore, India",18.1,4.0,11.6,20.5,68.7,NaN,NaN,NaN,NaN,51.8,NaN,60.2,https://www.numbeo.com/cost-of-living/in/Indor...
523,Guwahati,NaN,NaN,IND,India,"Guwahati, India",17.7,3.3,11.0,21.3,62.9,NaN,NaN,NaN,NaN,NaN,NaN,75.8,https://www.numbeo.com/cost-of-living/in/Guwah...
524,Lucknow,NaN,NaN,IND,India,"Lucknow (Lakhnau), India",17.5,3.4,11.0,19.8,69.8,NaN,NaN,NaN,NaN,54.5,62.9,77.9,https://www.numbeo.com/cost-of-living/in/Luckn...
525,Rajkot,NaN,NaN,IND,India,"Rajkot, India",17.1,3.2,10.6,19.9,51.9,NaN,NaN,NaN,NaN,NaN,NaN,59.8,https://www.numbeo.com/cost-of-living/in/Rajko...


In [33]:
df.to_pickle("./data/numbeo_links_2026.pkl")